# ChronosDecider — Demo

**Game Master agéntico en tiempo real**: Strands Decider 2B decide la rama narrativa; Amazon Bedrock la narra.

> Tags de la submission: `#strands-decider-2b` `#tool-selection`

Este notebook es una **demostración** del código del repositorio (módulos en `src/` y `decision_patterns/`). No reimplementa nada: importa el motor y muestra el flujo.

- **Sin servidor de Decider ni Bedrock configurados**, el motor degrada a backends locales (decisión simulada + narrativa de plantilla) para que la demo corra en cualquier parte.
- **Con `strands-decider serve` y Bedrock configurados**, las mismas celdas usan el modelo de decisión real y la narrativa generada.

Ejecútalo desde la **raíz del repositorio** para que las importaciones de `src` funcionen.

## 1. Preparación del entorno

Aseguramos que la raíz del repositorio esté en `sys.path` para poder importar `src` y `decision_patterns`.

In [ ]:
import sys, os

# Si el notebook se ejecuta desde notebooks/, subimos un nivel a la raíz del repo.
repo_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("Raíz del repo:", repo_root)

## 2. La decisión (Strands Decider 2B) y la generación (Bedrock)

El `DecisionEngine` orquesta todo: decide la rama con Decider 2B, aplica el triage de confianza (`<= 0.70` → Human-in-the-Loop) y, si la decisión es fiable, narra con Bedrock.

In [ ]:
from src.engine import DecisionEngine

engine = DecisionEngine()

def mostrar(result):
    print(f"Acción clasificada : {result.action_type}")
    print(f"Rama narrativa     : {result.target_branch}")
    print(f"Certeza            : {result.confidence_score:.1%}")
    print(f"Latencia decisión  : {result.decision_latency_ms:.2f} ms (objetivo < 100 ms, dentro = {result.within_latency_target})")
    print(f"Decisión vía       : {result.decision_source}")
    print(f"Narrativa vía      : {result.narrative_source}")
    print(f"Fallback           : {result.trigger_fallback}")
    print(f"Respuesta GM       : {result.narrative_response or '(sin narrar: se pidió aclaración)'}")
    if result.clarification_prompt:
        print(f"Aclaración         : {result.clarification_prompt}")

### 2a. Acción clara → alta confianza → se narra

Decider elige una rama con confianza por encima del umbral, así que el agente actúa y Bedrock (o la plantilla) narra.

In [ ]:
mostrar(engine.evaluate("atacar al goblin con mi espada"))

### 2b. Acción ambigua → baja confianza → guardrail (Human-in-the-Loop)

Aquí está el valor de la **confianza calibrada**: cuando Decider no está seguro, el agente NO inventa una narración; pide aclaración.

In [ ]:
mostrar(engine.evaluate("mmm no sé qué hacer"))

## 3. El patrón de decisión reutilizable (`#tool-selection`)

El entregable reutilizable de la hackathon vive en `decision_patterns/`. Es independiente del juego: construye una pregunta `choice` de Decider y devuelve la elección con su confianza, más un flag `should_act` que aplica el umbral como guardrail.

Las celdas siguientes muestran las **funciones puras** del patrón (construcción de la pregunta y parseo de la respuesta) sin necesidad de un servidor en marcha.

In [ ]:
from decision_patterns import build_choice_question, parse_choice_answer

# 1) La pregunta tipada que se le envía a Strands Decider 2B.
pregunta = build_choice_question(
    "¿Qué equipo debe atender este mensaje?",
    ["billing", "sales", "retail"],
)
pregunta

In [ ]:
# 2) Parseo de una respuesta del servidor (ejemplo): elección + confianza calibrada.
respuesta_servidor = {
    "answers": {"tool_choice": {"choice": "billing", "confidence": 0.83}},
    "latency_ms": 118.0,
}
eleccion = parse_choice_answer(
    respuesta_servidor, ["billing", "sales", "retail"], threshold=0.70, latency_ms=120.0
)
print("Herramienta elegida:", eleccion.tool)
print("Confianza          :", eleccion.confidence)
print("¿Actuar?           :", eleccion.should_act, "(si es False, el agente pide aclaración)")

### 3a. (Opcional) Contra el servidor real de Strands Decider

Si tienes el servidor en marcha:

```bash
pip install strands-decider
strands-decider serve StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8000
```

la siguiente celda hará una decisión **real**. Si el servidor no responde, lo indica sin romper el notebook.

In [ ]:
import urllib.error
from decision_patterns import ToolSelectionPattern

pattern = ToolSelectionPattern(
    instructions="¿Qué equipo debe atender este mensaje?",
    options=["billing", "sales", "retail"],
    confidence_threshold=0.70,
)

try:
    choice = pattern.select("¡Ayuda! Mis pagos llevan 3 días fallando.")
    print("Elección real:", choice.tool, "| confianza:", round(choice.confidence, 3),
          "| latencia:", round(choice.latency_ms, 1), "ms")
except (urllib.error.URLError, ConnectionError, TimeoutError, OSError) as exc:
    print("Servidor de Strands Decider no disponible:", exc)
    print("Levántalo con: strands-decider serve StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8000")

## 4. Telemetría

El motor registra, por cada decisión, la latencia y la confianza. Resumen agregado tras las evaluaciones de arriba:

In [ ]:
resumen = engine.telemetry.summary()
if resumen is None:
    print("Aún no hay registros de telemetría.")
else:
    print(f"Evaluaciones        : {resumen.total}")
    print(f"Latencia media/máx  : {resumen.avg_latency_ms:.2f} / {resumen.max_latency_ms:.2f} ms")
    print(f"Dentro del objetivo : {resumen.within_target_ratio:.0%} (< 100 ms)")
    print(f"Certeza media       : {resumen.avg_confidence:.1%}")
    print(f"Fallbacks activados : {resumen.fallback_count}")

## 5. Siguiente paso

- Demo interactiva completa por consola: `python main.py`
- Pasos de activación de Strands Decider 2B y Amazon Bedrock: ver el `README.md` del repositorio.
- Patrón reutilizable y su documentación: carpeta `decision_patterns/`.